# Import

In [1]:
!pip -q install transformers accelerate datasets sentencepiece

# Model

In [2]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# 더 가볍게 시작하려면:
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

print("Loaded:", MODEL_NAME)
print("Device:", model.device)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-3B-Instruct
Device: cuda:0


# Util Func

In [3]:
import re

def normalize_text(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def clean_answer(text: str) -> str:
    text = text.strip()
    # 첫 줄만 사용
    text = text.split("\n")[0].strip()
    # 코드블록 같은 이상 출력 잘라내기
    text = text.split("```")[0].strip()
    return text

def generate_text(prompt: str, max_new_tokens: int = 128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # temperature/top_p/top_k 안 씀
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

## QA Func

### With Context

In [4]:
def ask_qa_with_context(context: str, question: str, max_new_tokens: int = 48):
    prompt = f"""다음 문서를 읽고 질문에 답하세요.

규칙:
1. 답만 한 문장으로 짧게 출력
2. 설명하지 말 것
3. 코드블록 출력 금지

[문서]
{context}

[질문]
{question}

[답변]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)
    if "[답변]" in text:
        text = text.split("[답변]")[-1].strip()
    return clean_answer(text)



### Without Context

In [5]:
def ask_qa_without_context(question: str, max_new_tokens: int = 48):
    prompt = f"""다음 질문에 답하세요.

규칙:
1. 답만 한 문장으로 짧게 출력
2. 설명하지 말 것
3. 코드블록 출력 금지

[질문]
{question}

[답변]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)
    if "[답변]" in text:
        text = text.split("[답변]")[-1].strip()
    return clean_answer(text)

## Predict Search Func

In [6]:
def predict_need_search(question: str, context: str = "", max_new_tokens: int = 4):
    prompt = f"""당신의 임무는 외부 검색 필요 여부만 판단하는 것입니다.

규칙:
1. 최신 정보, 현재 시점 정보, 자주 바뀌는 사실이면 SEARCH
2. 일반 상식, 오래 변하지 않는 사실이면 NO_SEARCH
3. 반드시 SEARCH 또는 NO_SEARCH 중 하나만 출력
4. 설명 금지

[질문]
{question}

[문맥]
{context if context else "없음"}

[판단]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)

    if "[판단]" in text:
        result = text.split("[판단]")[-1].strip()
    else:
        result = text.strip()

    result = result.split()[0].upper()

    if result == "SEARCH":
        return "SEARCH"
    elif result == "NO_SEARCH":
        return "NO_SEARCH"
    return result

# Sample Test

In [7]:
context = "서울은 대한민국의 수도이며, 대한민국 최대의 도시이다."
question = "대한민국의 수도는 어디인가?"

print("=== 문서 있음 ===")
print(ask_qa_with_context(context, question))

print("\n=== 문서 없음 ===")
print(ask_qa_without_context(question))

print("\n=== SEARCH 판단 ===")
print("Q1:", predict_need_search("대한민국의 수도는 어디인가?"))   # 기대: NO_SEARCH
print("Q2:", predict_need_search("2026년 현재 대한민국 대통령은 누구인가?"))  # 기대: SEARCH
print("Q3:", predict_need_search("오늘 원달러 환율은 얼마인가?"))   # 기대: SEARCH
print("Q4:", predict_need_search("물의 화학식은 무엇인가?"))       # 기대: NO_SEARCH

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== 문서 있음 ===
서울

=== 문서 없음 ===
서울

=== SEARCH 판단 ===
Q1: NO_SEARCH
Q2: SEARCH
Q3: SEARCH
Q4: NO_SEARCH


# DataSet

In [8]:
from datasets import load_dataset

dataset = load_dataset("KorQuAD/squad_kor_v1", split="validation[:300]")

print(type(dataset))
print(dataset)

README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

<class 'datasets.arrow_dataset.Dataset'>
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})


In [9]:
print("=== Dataset Summary ===")
print(dataset)

print("\n=== Number of Rows ===")
print(len(dataset))

print("\n=== Column Names ===")
print(dataset.column_names)

print("\n=== First Sample ===")
print(dataset[0])

=== Dataset Summary ===
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})

=== Number of Rows ===
300

=== Column Names ===
['id', 'title', 'context', 'question', 'answers']

=== First Sample ===
{'id': '6548850-0-0', 'title': '임종석', 'context': '1989년 2월 15일 여의도 농민 폭력 시위를 주도한 혐의(폭력행위등처벌에관한법률위반)으로 지명수배되었다. 1989년 3월 12일 서울지방검찰청 공안부는 임종석의 사전구속영장을 발부받았다. 같은 해 6월 30일 평양축전에 임수경을 대표로 파견하여 국가보안법위반 혐의가 추가되었다. 경찰은 12월 18일~20일 사이 서울 경희대학교에서 임종석이 성명 발표를 추진하고 있다는 첩보를 입수했고, 12월 18일 오전 7시 40분 경 가스총과 전자봉으로 무장한 특공조 및 대공과 직원 12명 등 22명의 사복 경찰을 승용차 8대에 나누어 경희대학교에 투입했다. 1989년 12월 18일 오전 8시 15분 경 서울청량리경찰서는 호위 학생 5명과 함께 경희대학교 학생회관 건물 계단을 내려오는 임종석을 발견, 검거해 구속을 집행했다. 임종석은 청량리경찰서에서 약 1시간 동안 조사를 받은 뒤 오전 9시 50분 경 서울 장안동의 서울지방경찰청 공안분실로 인계되었다.', 'question': '임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배 된 날은?', 'answers': {'text': ['1989년 2월 15일'], 'answer_start': [0]}}


## rough match Func

In [10]:
def rough_match(pred: str, gold: str) -> bool:
    pred_n = normalize_text(pred)
    gold_n = normalize_text(gold)
    return (gold_n in pred_n) or (pred_n in gold_n)

# BaseLine Test With Context

In [11]:
correct = 0
total = 0
results = []

for i, ex in enumerate(dataset):
    if i % 20 == 0:
        print(f"Progress: {i}/{len(dataset)}")

    context = ex["context"]
    question = ex["question"]
    gold_answers = ex["answers"]["text"]
    gold = gold_answers[0] if len(gold_answers) > 0 else ""

    pred = ask_qa_with_context(context, question)

    ok = rough_match(pred, gold)
    correct += int(ok)
    total += 1

    results.append({
        "question": question,
        "gold": gold,
        "pred": pred,
        "correct": ok
    })

print(f"Progress: {len(dataset)}/{len(dataset)}")
print(f"With context accuracy (rough): {correct}/{total} = {correct/total:.3f}")
results[:3]

Progress: 0/300
Progress: 20/300
Progress: 40/300
Progress: 60/300
Progress: 80/300
Progress: 100/300
Progress: 120/300
Progress: 140/300
Progress: 160/300
Progress: 180/300
Progress: 200/300
Progress: 220/300
Progress: 240/300
Progress: 260/300
Progress: 280/300
Progress: 300/300
With context accuracy (rough): 263/300 = 0.877


[{'question': '임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배 된 날은?',
  'gold': '1989년 2월 15일',
  'pred': '1989년 2월 15일',
  'correct': True},
 {'question': '1989년 6월 30일 평양축전에 대표로 파견 된 인물은?',
  'gold': '임수경',
  'pred': '임수경',
  'correct': True},
 {'question': '임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배된 연도는?',
  'gold': '1989년',
  'pred': '1989년',
  'correct': True}]

In [12]:
wrong_cases = [r for r in results if not r["correct"]]
print(f"Wrong cases: {len(wrong_cases)}")

for i, case in enumerate(wrong_cases[:5]):
    print(f"\n[Wrong Case {i+1}]")
    print("Q:", case["question"])
    print("Gold:", case["gold"])
    print("Pred:", case["pred"])

Wrong cases: 37

[Wrong Case 1]
Q: 임종석을 검거한 장소는 경희대 내 어디인가?
Gold: 학생회관 건물 계단
Pred: 학생회관 계단

[Wrong Case 2]
Q: 미국 군대 내 두번째로 높은 직위는 무엇인가?
Gold: 미국 육군 부참모 총장
Pred: 美军参谋长联席会议主席

[Wrong Case 3]
Q: 미국 군대에서 두번째로 높은 직위는?
Gold: 미국 육군 부참모 총장
Pred: 장교 대장

[Wrong Case 4]
Q: 알렉산더 헤이그가 미국 육군사관학교로 임명받은 해는 언제인가?
Gold: 1944년
Pred: 1947년

[Wrong Case 5]
Q: 육군사관학교에서 졸업한 헤이그가 제일 처음 소위로 발령받은 부대는 무엇이었나?
Gold: 정통 제병 연합부대
Pred: 캔자스 주 포트라일리


# BaseLine Test Without Context

In [13]:
correct_no_ctx = 0
total_no_ctx = 0
results_no_ctx = []

for ex in dataset:
    question = ex["question"]
    gold_answers = ex["answers"]["text"]
    gold = gold_answers[0] if len(gold_answers) > 0 else ""

    pred = ask_qa_without_context(question)

    ok = rough_match(pred, gold)
    correct_no_ctx += int(ok)
    total_no_ctx += 1

    results_no_ctx.append({
        "question": question,
        "gold": gold,
        "pred": pred,
        "correct": ok
    })

print(f"Without context accuracy (rough): {correct_no_ctx}/{total_no_ctx} = {correct_no_ctx/total_no_ctx:.3f}")
results_no_ctx[:3]

Without context accuracy (rough): 12/300 = 0.040


[{'question': '임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배 된 날은?',
  'gold': '1989년 2월 15일',
  'pred': '2016년 11월 17일',
  'correct': False},
 {'question': '1989년 6월 30일 평양축전에 대표로 파견 된 인물은?',
  'gold': '임수경',
  'pred': '김영삼 전 대통령',
  'correct': False},
 {'question': '임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배된 연도는?',
  'gold': '1989년',
  'pred': '2016년',
  'correct': False}]

In [14]:
wrong_cases_no_ctx = [r for r in results_no_ctx if not r["correct"]]
print(f"Wrong cases without context: {len(wrong_cases_no_ctx)}")

for i, case in enumerate(wrong_cases_no_ctx[:5]):
    print(f"\n[Wrong No-Context Case {i+1}]")
    print("Q:", case["question"])
    print("Gold:", case["gold"])
    print("Pred:", case["pred"])

Wrong cases without context: 288

[Wrong No-Context Case 1]
Q: 임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배 된 날은?
Gold: 1989년 2월 15일
Pred: 2016년 11월 17일

[Wrong No-Context Case 2]
Q: 1989년 6월 30일 평양축전에 대표로 파견 된 인물은?
Gold: 임수경
Pred: 김영삼 전 대통령

[Wrong No-Context Case 3]
Q: 임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배된 연도는?
Gold: 1989년
Pred: 2016년

[Wrong No-Context Case 4]
Q: 임종석을 검거한 장소는 경희대 내 어디인가?
Gold: 학생회관 건물 계단
Pred: 경희대학교 법과대학 강당

[Wrong No-Context Case 5]
Q: 임종석이 조사를 받은 뒤 인계된 곳은 어딘가?
Gold: 서울지방경찰청 공안분실
Pred: 서울중앙지방검찰청 형사6부


## PipeLine

In [15]:
def qa_with_search_pipeline(question, context):
    decision = predict_need_search(question, context)

    if decision == "SEARCH":
        pred = ask_qa_with_context(context, question)
    else:
        pred = ask_qa_without_context(question)

    return decision, pred

### PipeLine Test

In [16]:
pipeline_correct = 0
pipeline_results = []

for ex in dataset:
    question = ex["question"]
    context = ex["context"]
    gold_answers = ex["answers"]["text"]
    gold = gold_answers[0] if len(gold_answers) > 0 else ""

    decision, pred = qa_with_search_pipeline(question, context)

    ok = rough_match(pred, gold)
    pipeline_correct += int(ok)

    pipeline_results.append({
        "question": question,
        "decision": decision,
        "gold": gold,
        "pred": pred,
        "correct": ok
    })

pipeline_acc = pipeline_correct / len(dataset)
print(f"Pipeline accuracy: {pipeline_correct}/{len(dataset)} = {pipeline_acc:.3f}")

Pipeline accuracy: 250/300 = 0.833


### PipeLine Errors

In [17]:
pipeline_errors = [r for r in pipeline_results if not r["correct"]]
print(f"Pipeline wrong cases: {len(pipeline_errors)}")

for i, case in enumerate(pipeline_errors[:10]):
    print(f"\n[Pipeline Error {i+1}]")
    print("Q:", case["question"])
    print("Decision:", case["decision"])
    print("Gold:", case["gold"])
    print("Pred:", case["pred"])

Pipeline wrong cases: 50

[Pipeline Error 1]
Q: 임종석을 검거한 장소는 경희대 내 어디인가?
Decision: SEARCH
Gold: 학생회관 건물 계단
Pred: 학생회관 계단

[Pipeline Error 2]
Q: 미국 군대 내 두번째로 높은 직위는 무엇인가?
Decision: SEARCH
Gold: 미국 육군 부참모 총장
Pred: 美军参谋长联席会议主席

[Pipeline Error 3]
Q: 미국 군대에서 두번째로 높은 직위는?
Decision: SEARCH
Gold: 미국 육군 부참모 총장
Pred: 장교 대장

[Pipeline Error 4]
Q: 알렉산더 헤이그가 미국 육군사관학교로 임명받은 해는 언제인가?
Decision: SEARCH
Gold: 1944년
Pred: 1947년

[Pipeline Error 5]
Q: 헤이그가 공부한 대학교는?
Decision: NO_SEARCH
Gold: 노터데임 대학교
Pred: 미국 스탠퍼드 대

[Pipeline Error 6]
Q: 알렉산더 헤이그가 나온 대학교는?
Decision: NO_SEARCH
Gold: 노터데임 대학교
Pred: 约翰斯·霍普金斯大学

[Pipeline Error 7]
Q: 육군사관학교에서 졸업한 헤이그가 제일 처음 소위로 발령받은 부대는 무엇이었나?
Decision: SEARCH
Gold: 정통 제병 연합부대
Pred: 캔자스 주 포트라일리

[Pipeline Error 8]
Q: 제럴드 포드 대통령 시기 헤이그가 최고사령부의 최고 사령관으로 임명된 곳은 어디인가?
Decision: SEARCH
Gold: 미국 유럽 연합군
Pred: 나토에서

[Pipeline Error 9]
Q: 퇴역 후 헤이그는 어느 회사의 대표가 되었나?
Decision: SEARCH
Gold: 미국 기술 주식 회사
Pred: AMC 엔터테인먼트

[Pipeline Error 10]
Q: 노아의 방주에 대해 기록하고있는 복음서는 무엇인가?
Decision: NO_S

In [18]:
under_search = []
qa_failure = []
over_search = []

for r in pipeline_results:
    if not r["correct"]:
        if r["decision"] == "NO_SEARCH":
            under_search.append(r)
        elif r["decision"] == "SEARCH":
            qa_failure.append(r)

print("UNDER_SEARCH:", len(under_search))
print("QA_FAILURE:", len(qa_failure))

UNDER_SEARCH: 15
QA_FAILURE: 35


# SEARCH / NO_SEARCH baseline용 소규모 수작업 평가

In [19]:
search_eval = [
    # ----------------------------
    # NO_SEARCH: 일반 상식 / 고정 지식 / 문맥으로 바로 답 가능
    # ----------------------------
    {"question": "대한민국의 수도는 어디인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "물의 화학식은 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "조선의 건국자는 누구인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "세종대왕은 어느 왕조의 왕인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "삼권분립이란 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "광합성은 무엇을 만드는 과정인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "대한민국의 국기는 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "지구는 태양계에서 몇 번째 행성인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "피타고라스 정리는 어떤 관계를 설명하는가?", "context": "", "label": "NO_SEARCH"},
    {"question": "한글을 창제한 왕은 누구인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "프랑스의 수도는 어디인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "뉴턴의 제2법칙은 무엇을 설명하는가?", "context": "", "label": "NO_SEARCH"},
    {"question": "DNA는 무엇의 약자인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "대한민국의 국화는 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "아인슈타인이 제안한 대표 이론은 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "태양은 어느 은하에 속해 있는가?", "context": "", "label": "NO_SEARCH"},
    {"question": "원주율을 보통 어떤 기호로 나타내는가?", "context": "", "label": "NO_SEARCH"},
    {"question": "대한민국의 공식 언어는 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "미국의 수도는 어디인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "산소의 원소기호는 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "이순신은 어느 시대의 인물인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "컴퓨터에서 CPU는 무엇의 약자인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "태평양은 대서양보다 넓은가?", "context": "", "label": "NO_SEARCH"},
    {"question": "조류는 알을 낳는가?", "context": "", "label": "NO_SEARCH"},
    {"question": "이 문서에서 저자가 주장한 핵심은 무엇인가?", "context": "본 논문은 small LLM에서 search policy distillation 가능성을 탐구한다.", "label": "NO_SEARCH"},

    # ----------------------------
    # SEARCH: 최신 정보 / 변동 정보 / 외부 확인 필요
    # ----------------------------
    {"question": "2026년 현재 대한민국 대통령은 누구인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 원달러 환율은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "이번 주 삼성전자 종가는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 서울 날씨는 어떤가?", "context": "", "label": "SEARCH"},
    {"question": "2025년 노벨문학상 수상자는 누구인가?", "context": "", "label": "SEARCH"},
    {"question": "이번 시즌 EPL 우승팀은 누구인가?", "context": "", "label": "SEARCH"},
    {"question": "현재 비트코인 가격은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 금 가격은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "이번 주 코스피 지수는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 뉴욕 증시 다우지수는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "현재 애플 시가총액은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "이번 달 한국은행 기준금리는 몇 퍼센트인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 서울의 미세먼지 농도는 어떤가?", "context": "", "label": "SEARCH"},
    {"question": "올해 아카데미 작품상 수상작은 무엇인가?", "context": "", "label": "SEARCH"},
    {"question": "현재 테슬라 주가는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 S&P500 지수는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "이번 주 국제 유가(WTI)는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "지금 한국의 실업률은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "현재 엔비디아 CEO는 누구인가?", "context": "", "label": "SEARCH"},
    {"question": "올해 UEFA 챔피언스리그 우승팀은 누구인가?", "context": "", "label": "SEARCH"},

    # ----------------------------
    # 경계/애매한 케이스: search 필요성 판단이 헷갈릴 수 있는 문제
    # ----------------------------
    {"question": "대한민국의 인구는 몇 명인가?", "context": "", "label": "SEARCH"},
    {"question": "서울의 면적은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "현재 가장 높은 빌딩은 무엇인가?", "context": "", "label": "SEARCH"},
    {"question": "대한민국의 최저임금은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 날짜는 무엇인가?", "context": "", "label": "SEARCH"},
]

In [20]:
pred_correct = 0
search_results = []

for ex in search_eval:
    pred = predict_need_search(ex["question"], ex["context"])
    ok = pred == ex["label"]
    pred_correct += int(ok)

    search_results.append({
        "question": ex["question"],
        "gold": ex["label"],
        "pred": pred,
        "correct": ok
    })

    print({
        "question": ex["question"],
        "gold": ex["label"],
        "pred": pred,
        "correct": ok
    })

print(f"\nSearch decision accuracy: {pred_correct}/{len(search_eval)} = {pred_correct/len(search_eval):.3f}")

{'question': '대한민국의 수도는 어디인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '물의 화학식은 무엇인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '조선의 건국자는 누구인가?', 'gold': 'NO_SEARCH', 'pred': 'SEARCH', 'correct': False}
{'question': '세종대왕은 어느 왕조의 왕인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '삼권분립이란 무엇인가?', 'gold': 'NO_SEARCH', 'pred': 'SEARCH', 'correct': False}
{'question': '광합성은 무엇을 만드는 과정인가?', 'gold': 'NO_SEARCH', 'pred': 'SEARCH', 'correct': False}
{'question': '대한민국의 국기는 무엇인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '지구는 태양계에서 몇 번째 행성인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '피타고라스 정리는 어떤 관계를 설명하는가?', 'gold': 'NO_SEARCH', 'pred': 'SEARCH', 'correct': False}
{'question': '한글을 창제한 왕은 누구인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '프랑스의 수도는 어디인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': T

In [21]:
wrong_search = [r for r in search_results if not r["correct"]]
print(f"Wrong search-decision cases: {len(wrong_search)}")

for i, case in enumerate(wrong_search):
    print(f"\n[Wrong Search Case {i+1}]")
    print("Q:", case["question"])
    print("Gold:", case["gold"])
    print("Pred:", case["pred"])

Wrong search-decision cases: 11

[Wrong Search Case 1]
Q: 조선의 건국자는 누구인가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 2]
Q: 삼권분립이란 무엇인가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 3]
Q: 광합성은 무엇을 만드는 과정인가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 4]
Q: 피타고라스 정리는 어떤 관계를 설명하는가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 5]
Q: 뉴턴의 제2법칙은 무엇을 설명하는가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 6]
Q: 아인슈타인이 제안한 대표 이론은 무엇인가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 7]
Q: 태양은 어느 은하에 속해 있는가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 8]
Q: 원주율을 보통 어떤 기호로 나타내는가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 9]
Q: 미국의 수도는 어디인가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 10]
Q: 태평양은 대서양보다 넓은가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 11]
Q: 이 문서에서 저자가 주장한 핵심은 무엇인가?
Gold: NO_SEARCH
Pred: SEARCH


# Save

In [22]:
import json

with open("qa_with_context_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

with open("qa_without_context_results.json", "w", encoding="utf-8") as f:
    json.dump(results_no_ctx, f, ensure_ascii=False, indent=2)

print("Saved result files.")

Saved result files.
